## Exercise 5: Geospatial wrangling and making maps

Skills: 
* More geospatial practice building on earlier skills
* Make a map with `folium` or `ipyleaflet` using `shared_utils.map_utils`

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [ ]:
import geopandas as gpd
import intake
import pandas as pd

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

from shapely.geometry import Polygon, LineString, Point 

import shared_utils
import altair as alt
from shared_utils import altair_utils 

import branca
# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command


## Research Question: What's the average number of trips per stop by operators in southern California? Show visualizations at the operator and county-level.
<br>**Geographic scope:** southern California counties
<br>**Deliverables:** chart(s) and map(s) showing metrics comparing across counties and also across operators. Make these visualizations using function(s).

### Prep data

* Use the same query, but grab a different set of operators. These are in southern California, so the map should zoom in counties ranging from LA to SD.
* To find what ITP IDs are what operators:
[agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* *Hint*: for some counties, there are multiple operators. Make sure the average trips per stop by counties is the weighted average.
* Use the same [shapefile for CA counties](https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12) as in Exercise 4.
* Join the data and only keep counties that have bus stops.


In [ ]:
#choosing a different set of ITP IDS
ITP_ID = [182, 183, 278, 154, 235, 232]

In [ ]:
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

In [ ]:
#creating point geometry
stops = shared_utils.geography_utils.create_point_geometry(stops, 'stop_lon', 'stop_lat')

In [ ]:
#bringing in CA dataframe
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson').to_crs(epsg=4326)

In [ ]:
#converting crs 
stops = stops.to_crs(epsg=4326)

In [ ]:
geojson.crs

In [ ]:
#Joining the 2 dataframes, only keeping stops that are in a county
stops_joined = gpd.sjoin(stops, geojson, how='inner')

### Bring in a new table from BigQuery - QUESTION most of the columns are null? 

* In `transitstacks`, there's a table called `ntd_stats`. 
* Decide what columns to keep.
* Merge `ntd_stats` with the `stops` table to create 1 df.

In [ ]:
ntd_stats = (tbl.transitstacks.ntd_stats()
             >> select(_.transit_provider, _.itp_id, _.modes, _.upt_total_2019)
             >> collect()
             >> filter(_.itp_id.isin(ITP_ID))
            )

In [ ]:
ntd_stats

### Merging the 2-  QUESTION 
* Make sure all stop ids are unique

In [ ]:
df2 = stops_joined.merge(ntd_stats, how = 'left', on ='itp_id')

In [ ]:
df2.shape

In [ ]:
df2.head(5)

### Aggregate - Help, how to get trips per stop?
<b> Instructions </b> 
* Write a function to aggregate to the operator level or county level, add new columns for desired metrics.
* Merge in CA shapefile to get a gdf.
* Add another `geometry` column, called `centroid`, and grab the county's centroid.
* Refer to [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.set_geometry.html) to see how to pick which column to use as the `geometry` for the gdf, since technically, a gdf can handle multiple geometry columns.



In [ ]:
#subset for count of stop ids by county.
subset1 = df2[['COUNTY_NAME','stop_id','geometry']]

In [ ]:
def aggregation(df, aggregated_column):
    df = df.dissolve(by=aggregated_column, aggfunc ='nunique').reset_index()
    return df 

In [ ]:
county_stops = aggregation(subset1, 'COUNTY_NAME')

In [ ]:
#subset for total  service vehicles by county
county_vehicles = aggregation(subset1, 'COUNTY_NAME')

In [ ]:
#combine the 2 metrics together
county = gpd.sjoin(county_vehicles, county_stops, how='inner')

In [ ]:
county = county.drop(columns = ['COUNTY_NAME_right','index_right'])

In [ ]:
county = county.rename(columns = {'COUNTY_NAME_left': 'COUNTY_NAME', 'stop_id':'count_of_unique_stops','servvehicles_2019':'total_service_vehicles'})

In [ ]:
county

In [ ]:
#merge again for polygon
county_geo  = pd.merge(county.drop(columns="geometry"), geojson,  on="COUNTY_NAME")

In [ ]:
#filter out the islands
county_geo = county_geo.loc[county_geo['ISLAND'] != 'Channel Islands'] 

In [ ]:
#add centroid for each county 
county_geo['centroid'] = county_geo.geometry.centroid

In [ ]:
county_geo.head(5)

In [ ]:
test = county_geo.drop(columns = ['centroid'])

### Visualizations
* Make one chart for comparing trips per stop by operators, and another chart for comparing it by counties. Use a function to do this.
* Make 1 map for comparing trips per stop by counties. Use `shared_utils.map_utils` to do this.
* Visualizations should follow the Cal-ITP style guide.
* More on `folium` and `ipyleaflet`: https://github.com/jorisvandenbossche/geopandas-tutorial/blob/master/05-more-on-visualization.ipynb

#### Bar chart
* Make it into a function.
* Don't print out bar chart b/c this causes a file saving error

In [ ]:
def bar_chart(df, x_col, y_col):
    y_col_stripped = y_col.replace('_',' ')
    chart_title = f"{y_col_stripped}" 
    chart = (alt.Chart(df).mark_bar().encode(
                 x=alt.X(x_col, title=x_col),
                 y=alt.Y(y_col, title=y_col_stripped),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.FIVETHIRTYEIGHT_CATEGORY_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250,
                 title = chart_title)
            )
    #return chart
    display(chart)

In [ ]:
bar_chart(county, 'COUNTY_NAME', 'stop_id_left') 

In [ ]:
bar_chart(county, 'COUNTY_NAME', 'stop_id_left') 

#### Map - HELP
* Object of type Point is not JSON serializable

In [ ]:
choropleth_dict = ({
            "layer_name": 'my layer',
            "MIN_VALUE": int(0),
            "MAX_VALUE": int(100),
            "plot_col_name": 'County_Name',
            "fig_width": '100%',
            "fig_height": '100%',
            "fig_min_width_px": '600px',
            "fig_min_height_px": '600px'})

In [ ]:
colorscale = branca.colormap.StepColormap(
                colors=["gray", "green", "navy"], 
                #index=[2_000, 4_000, 8_000],
                #vmin=0, vmax=15_000,
)


In [ ]:
def grab_region_centroids():
    # This parquet is created in shared_utils/shared_data.py
    df = pd.read_parquet(
        "gs://calitp-analytics-data/data-analyses/ca_county_centroids.parquet")
    
    df = df.assign(
        centroid = df.centroid.apply(lambda x: x.tolist())
    )    
    
    # Manipulate parquet file to be dictionary to use in map_utils
    region_centroids = dict(
        zip(df.county_name, 
            df[["centroid", "zoom"]].to_dict(orient="records")
           )
    )

    return region_centroids

In [ ]:
REGION_CENTROIDS = grab_region_centroids()

In [ ]:
#shared_utils.map_utils.make_ipyleaflet_choropleth_map(test, 
                                                     # plot_col = 'stop_id_left',
                                                     # geometry_col = 'COUNTY_NAME', 
                                                     # choropleth_dict = choropleth_dict, 
                                                      #colorscale = colorscale,
                                                     # zoom=REGION_CENTROIDS["CA"]["zoom"], centroid = REGION_CENTROIDS["CA"]["centroid"])